In [1]:
import os

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
import bs4
import requests
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from dotenv import load_dotenv, dotenv_values

load_dotenv(override=True)
env_dict = dotenv_values(".env")
print("--- Contents of .env file ---")
for key, value in env_dict.items():
    # Masking the value so you don't accidentally print your full API key to your screen!
    masked_value = f"{value[:5]}...{value[-4:]}" if len(value) > 10 else value
    print(f"{key}: {masked_value}")

--- Contents of .env file ---
OPENAI_API_KEY: sk-pr...-PAA
GOOGLE_API_KEY: AQ.Ab...dTQA
TAVILY_API_KEY: tvly-...keLk
USER_AGENT: rag-f.../1.0
LANGSMITH_TRACING: true
LANGSMITH_ENDPOINT: https....com
LANGSMITH_API_KEY: lsv2_...2276
LANGSMITH_PROJECT: langc...demy
HF_TOKEN: hf_vG...sddc
LANGCHAIN_TRACING_V2: true
LANGCHAIN_API_KEY: lsv2_...13cd
LANGCHAIN_PROJECT: langc...demo


/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/4273366485.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [2]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
    )
),
)
blog_docs = loader.load()

#split
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=50
)

#make splits
splits = text_splitter.split_documents(blog_docs)

#index
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import Chroma 

# vectorstore = Chroma.from_documents(
#     documents=splits,
#     embedding=GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001"))

hf_embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
# 3. Create your Chroma vector store
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=hf_embeddings
)
retriever = vectorstore.as_retriever()



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspectives = ChatPromptTemplate.from_template(template)

# 2. Build the chain (Swapped ChatOpenAI for ChatGoogleGenerativeAI)

local_llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model", # LM Studio automatically uses whatever model is loaded in the server
    temperature=0
)

generate_queries = (
    prompt_perspectives 
    | local_llm 
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

# 4. Run it!
questions = generate_queries.invoke({"question": "What is an LLM agent?"})
print(questions)

['Explain the concept of an LLM agent in detail.', 'How do Large Language Model agents function or operate?', 'What are some real-world use cases or applications for implementing LLM agents?', 'Describe the technical architecture and core components (such as planning, memory, or tool usage) that define an LLM agent.', 'What capabilities allow an AI system to autonomously complete complex, multi-step tasks?']


In [9]:
from langchain_core.load import dumps, loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

#retrieve 
question = "what is task decomposition for llm agents"
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question":question})
len(docs)

/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/997358273.py:10: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]
/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/997358273.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


8

In [11]:
from operator import itemgetter
from langchain_openai import ChatOpenAI
from langchain_core.runnables import RunnablePassthrough

template = """Answer the following question based on this context:

{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(
    base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model", # LM Studio automatically uses whatever model is loaded in the server
    temperature=0
)

final_rag_chain = (
    {"context": retrieval_chain, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})


/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/997358273.py:10: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  return [loads(doc) for doc in unique_docs]


''

In [12]:
from langchain_core.prompts import ChatPromptTemplate

# RAG-Fusion: Related
template = """You are a helpful assistant that generates multiple search queries based on a single input query. \n
Generate multiple search queries related to: {question} \n
Output (4 queries):"""
prompt_rag_fusion = ChatPromptTemplate.from_template(template)

In [16]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_queries = (
    prompt_rag_fusion 
    | ChatOpenAI(base_url="http://127.0.0.1:1234/v1/", 
    api_key="lm-studio", 
    model="local-model", # LM Studio automatically uses whatever model is loaded in the server
    temperature=0)
    | StrOutputParser() 
    | (lambda x: x.split("\n"))
)

In [17]:
from langchain_core.load import dumps, loads

def reciprocal_rank_fusion(results: list[list], k=60):
    """ Reciprocal_rank_fusion that takes multiple lists of ranked documents 
        and an optional parameter k used in the RRF formula """
    
    # Initialize a dictionary to hold fused scores for each unique document
    fused_scores = {}

    # Iterate through each list of ranked documents
    for docs in results:
        # Iterate through each document in the list, with its rank (position in the list)
        for rank, doc in enumerate(docs):
            # Convert the document to a string format to use as a key (assumes documents can be serialized to JSON)
            doc_str = dumps(doc)
            # If the document is not yet in the fused_scores dictionary, add it with an initial score of 0
            if doc_str not in fused_scores:
                fused_scores[doc_str] = 0
            # Retrieve the current score of the document, if any
            previous_score = fused_scores[doc_str]
            # Update the score of the document using the RRF formula: 1 / (rank + k)
            fused_scores[doc_str] += 1 / (rank + k)

    # Sort the documents based on their fused scores in descending order to get the final reranked results
    reranked_results = [
        (loads(doc), score)
        for doc, score in sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
    ]

    # Return the reranked results as a list of tuples, each containing the document and its fused score
    return reranked_results

retrieval_chain_rag_fusion = generate_queries | retriever.map() | reciprocal_rank_fusion
docs = retrieval_chain_rag_fusion.invoke({"question": question})
len(docs)

/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/2443664460.py:26: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


11

In [19]:
from langchain_core.runnables import RunnablePassthrough

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

final_rag_chain = (
    {"context": retrieval_chain_rag_fusion, 
     "question": itemgetter("question")} 
    | prompt
    | llm
    | StrOutputParser()
)

final_rag_chain.invoke({"question":question})

/var/folders/7t/_8zkzcdx12d70lrtsb5dgnrm0000gn/T/ipykernel_22957/2443664460.py:26: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  (loads(doc), score)


'Based on the context, **task decomposition** for LLM agents is the process of breaking down a large, complex task into smaller, manageable subgoals or steps.\n\nThis capability allows the agent to handle complexity efficiently and improve its performance when tackling difficult problems.\n\n### Key Concepts and Methods:\n\n1.  **Core Function:** The primary goal is to transform a single, overwhelming objective into multiple simpler tasks that can be addressed sequentially or in parallel.\n2.  **Chain of Thought (CoT):** This is a standard technique where the model is prompted to "think step by step." CoT utilizes extra computation time to decompose hard tasks into smaller, simpler steps, thereby illuminating the model\'s reasoning process.\n3.  **Tree of Thoughts (ToT):** This extends CoT by exploring multiple potential reasoning paths at each step. Instead of a linear chain, it generates multiple thoughts per step, creating a tree structure that allows for more comprehensive problem-